# Day 19 — Pandas Deep Dive
## Advanced Filtering, Aggregation & Reporting

Production analytics using date filtering, rolling averages,  
crosstab analysis, and executive summary reporting.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("sample_data.csv")
df["date"]         = pd.to_datetime(df["date"])
df["availability"] = df["actual_run"] / df["planned"]
df["performance"]  = df["units"] / (df["planned"] * 1.8)
df["quality"]      = df["good_units"] / df["units"]
df["oee"]          = df["availability"] * df["performance"] * df["quality"]

print(f"Dataset: {df.shape[0]} rows | Date range: "
      f"{df['date'].min().date()} to {df['date'].max().date()}")

Dataset: 240 rows | Date range: 2024-01-01 to 2024-01-30


## Date-Based Filtering

Filtering production records by week to support  
rolling operational reviews.

In [2]:
# First week
first_week = df[(df["date"] >= "2024-01-01") &
                (df["date"] <= "2024-01-07")]

# Last 7 days of dataset
latest  = df["date"].max()
last_7  = df[df["date"] >= latest - pd.Timedelta(days=7)]

print(f"First week records : {len(first_week)}")
print(f"Last 7 days records: {len(last_7)}")
print(f"\nFirst week OEE by machine:")
print(first_week.groupby("machine")["oee"].mean().round(3))

First week records : 56
Last 7 days records: 64

First week OEE by machine:
machine
Assembly Line A    0.779
CNC Mill #4        0.783
Press #2           0.780
Weld Station B     0.836
Name: oee, dtype: float64


## Fleet Executive Summary

Ranked performance table with average OEE, variability,  
and classification by machine center.

In [3]:
def build_executive_summary(df):
    summary = df.groupby("machine").agg(
        avg_oee      = ("oee", "mean"),
        std_oee      = ("oee", "std"),
        min_oee      = ("oee", "min"),
        total_units  = ("units", "sum"),
        record_count = ("oee", "count")
    ).reset_index()

    summary["status"] = summary["avg_oee"].apply(
        lambda x: "World Class"    if x >= 0.85
             else "Acceptable"     if x >= 0.70
             else "Below Threshold"
    )
    summary["rank"]        = summary["avg_oee"].rank(ascending=False).astype(int)
    summary["Avg OEE %"]   = (summary["avg_oee"] * 100).round(1)
    summary["Std Dev %"]   = (summary["std_oee"] * 100).round(1)
    summary["Min OEE %"]   = (summary["min_oee"] * 100).round(1)

    return summary.sort_values("rank")[[
        "rank", "machine", "Avg OEE %",
        "Std Dev %", "Min OEE %",
        "total_units", "record_count", "status"
    ]]

exec_summary = build_executive_summary(df)
print(exec_summary.to_string(index=False))

 rank         machine  Avg OEE %  Std Dev %  Min OEE %  total_units  record_count     status
    1 Assembly Line A       80.9        9.0       61.4        47582            60 Acceptable
    2  Weld Station B       80.6        9.8       60.5        46523            60 Acceptable
    3     CNC Mill #4       80.5       11.1       63.4        47522            60 Acceptable
    4        Press #2       80.2        8.6       61.4        47009            60 Acceptable


## Defect Code Analysis

Crosstab showing defect code distribution by machine center.  
Normalized view reveals relative defect exposure per machine.

In [ ]:
# Raw counts
defect_counts = pd.crosstab(df["machine"], df["defect_code"])
print("Defect Code Counts by Machine:")
print(defect_counts)

print()

# Normalized
defect_pct = pd.crosstab(
    df["machine"],
    df["defect_code"],
    normalize="index"
).round(3)
print("Defect Code Distribution (%) by Machine:")
print(defect_pct)

NameError: name 'python' is not defined

## Rolling OEE Trend — CNC Mill #4

7-day rolling average smooths daily variation to reveal  
underlying performance trend.

In [ ]:
cnc = df[df["machine"] == "CNC Mill #4"].copy()
cnc_daily = cnc.groupby("date")["oee"].mean().reset_index()
cnc_daily.columns = ["date", "daily_oee"]
cnc_daily["rolling_7d"] = cnc_daily["daily_oee"].rolling(window=7).mean()

plt.figure(figsize=(12, 5))
plt.plot(cnc_daily["date"], cnc_daily["daily_oee"],
         color="lightsteelblue", linewidth=1.5,
         marker="o", markersize=3, label="Daily OEE")
plt.plot(cnc_daily["date"], cnc_daily["rolling_7d"],
         color="steelblue", linewidth=2.5, label="7-Day Rolling Avg")
plt.axhline(y=0.85, color="red", linestyle="--",
            linewidth=1.5, label="World Class")
plt.axhline(y=0.70, color="orange", linestyle="--",
            linewidth=1.5, label="Minimum Acceptable")
plt.title("CNC Mill #4 — OEE Trend with 7-Day Rolling Average",
          fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("OEE")
plt.ylim(0, 1.0)
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig("oee_rolling_trend.png", dpi=150)
plt.show()

## Key Findings

- First week vs last 7 days comparison enables trend detection
- Executive summary ranks all machines by average OEE with variability metrics
- Crosstab reveals defect code distribution per machine center
- 7-day rolling average filters daily noise to expose true performance trend

*High std deviation = process instability — priority for investigation*